In [26]:
import numpy as np
import pyvista as pv
from scipy.spatial import KDTree
import sys
sys.path.append("..")
from error_func import error

case_name = "case2"
mesh_name = "coil_box"
solver1 = "ngsolve"
solver2 = "moose"

sol_ngsolve = pv.read(f"../../output/{case_name}/{case_name}_{solver1}.vtu")
sol_moose = pv.read(f"../../output/{case_name}/{case_name}_{solver2}.vtu")

In [27]:
tree = KDTree(sol_ngsolve.points)
distances, indices = tree.query(sol_moose.points)

max_dist = np.max(distances)
print(f"Maximum alignment error (distance): {max_dist:.6e}")
if max_dist > 1e-4:
    print("Warning: Large distance detected. Are the geometries identical?")

aligned_grid = sol_moose.copy()


for array_name in sol_ngsolve.point_data.keys():
        data = sol_ngsolve.point_data[array_name]
        
        reordered_data = data[indices]
        
        aligned_grid.point_data[array_name] = reordered_data
        print(f"Transferred array: {array_name}")

aligned_grid.save(f"../../output/{case_name}/{case_name}_{solver1}_reordered_to_{solver2}.vtu")

Maximum alignment error (distance): 8.396113e-11
Transferred array: magnetic_vector_potential_nd
Transferred array: magnetic_flux_density
Transferred array: current_density
Transferred array: magnetic_vector_potential
Transferred array: electric_potential


In [28]:
sol_ngsolve = pv.read(f"../../output/{case_name}/{case_name}_{solver1}_reordered_to_{solver2}.vtu")
# elec_pot_ngsolve = sol_ngsolve["electric_potential"]
mag_flux_ngsolve = sol_ngsolve["magnetic_flux_density"]
mag_vec_ngsolve = sol_ngsolve["magnetic_vector_potential"]

sol_moose = pv.read(f"../../output/{case_name}/{case_name}_{solver2}.vtu")
# elec_pot_moose = sol_moose["electric_potential"]
mag_flux_moose = sol_moose["magnetic_flux_density"]
mag_vec_moose = sol_moose["magnetic_vector_potential"]

# print(elec_pot_ngsolve.shape)
# print(elec_pot_moose.shape)

print(mag_flux_ngsolve.shape)
print(mag_flux_moose.shape)

print(mag_vec_ngsolve.shape)
print(mag_vec_moose.shape)

(278516, 3)
(278516, 3)
(278516, 3)
(278516, 3)


In [29]:
mesh = sol_moose.copy()

mesh.point_data.remove("magnetic_vector_potential")
# mesh.point_data.remove("electric_potential")
mesh.point_data.remove("magnetic_flux_density")

In [30]:
# print(f"Electric potential errors between {solver1} and {solver2}:")

# mesh = error(sol=elec_pot_ngsolve, sol_ref=elec_pot_moose, 
#              eps = 1e-6, mesh=mesh, tag="scalar", save_tag="V")

In [31]:
print(f"Magnetic flux density errors between {solver1} and {solver2}:")

mesh = error(sol=mag_flux_ngsolve, sol_ref=mag_flux_moose, 
             eps = 1e-6, mesh=mesh, tag="vector", save_tag="A")

Magnetic flux density errors between ngsolve and moose:

  * Max. absolute error in x direction  : 1.107e-01.
  * Avg. absolute error in x direction  : 7.252e-03.

  * Max. relative error in x direction : 3.903e+06 %.
  * Avg. relative error in x direction : 3.046e+02 %.

  * Max. absolute error in y direction  : 1.059e-01.
  * Avg. absolute error in y direction  : 5.407e-03.

  * Max. relative error in y direction : 6.417e+05 %.
  * Avg. relative error in y direction : 4.215e+02 %.

  * Max. absolute error in z direction  : 1.383e-01.
  * Avg. absolute error in z direction  : 1.232e-02.

  * Max. relative error in z direction : 1.111e+06 %.
  * Avg. relative error in z direction : 6.874e+02 %.


In [32]:
print(f"Magnetic vector potential errors between {solver1} and {solver2}:")

mesh = error(sol=mag_vec_ngsolve, sol_ref=mag_vec_moose, 
             eps = 1e-6, mesh=mesh, tag="vector", save_tag="B")

Magnetic vector potential errors between ngsolve and moose:

  * Max. absolute error in x direction  : 5.390e-03.
  * Avg. absolute error in x direction  : 2.495e-04.

  * Max. relative error in x direction : 6.600e+04 %.
  * Avg. relative error in x direction : 5.808e+02 %.

  * Max. absolute error in y direction  : 9.467e-03.
  * Avg. absolute error in y direction  : 5.043e-04.

  * Max. relative error in y direction : 2.732e+04 %.
  * Avg. relative error in y direction : 4.951e+02 %.

  * Max. absolute error in z direction  : 4.076e-03.
  * Avg. absolute error in z direction  : 1.348e-04.

  * Max. relative error in z direction : 1.120e+05 %.
  * Avg. relative error in z direction : 7.368e+02 %.


In [33]:
mesh.save(f"../../output/{case_name}/{case_name}_error_ngsolve_moose.vtu")